# Generated-batch exploratory data analysis

This notebook performs bounded, read-only scientific inspection of completed generated COMSOL batches before model training. It validates the same source contracts as the direct dataset builder and materializes only an explicit ordered prefix.

## Data-domain boundary

The workflow reads only generation metadata, terminal manifests, raw case exports, processed reference solutions, and optional solve timing below GENERATED_DATA_ROOT. It does not open final training datasets, splits, normalizers, runs, checkpoints, evaluation caches, or W&B state.

In [ ]:
import pandas as pd
from IPython.display import Markdown
from IPython.display import display as show
from src import common, domain
from src.analysis import eda

## Batch selection and safety limit

The maintained defaults compare two representative batches and materialize at most 100 manifest-ordered cases from each. Edit BATCH_NAMES and MAX_CASES deliberately before requesting a broader comparison.


In [ ]:
BATCH_NAMES = ["lhs_var80_seed3001", "lhs_var120_seed4001"]
MAX_CASES = 100

In [ ]:
task = domain.tasks.registry.get_task("steady_flow")
GENERATED_DATA_ROOT = common.paths.get_generated_data_root()
GENERATION_META_ROOT = common.paths.get_generation_meta_root()
GENERATION_RAW_ROOT = common.paths.get_generation_raw_root()
GENERATION_PROCESSED_ROOT = common.paths.get_generation_processed_root()

generation_paths = (
    pd.Series(
        {
            "root": str(GENERATED_DATA_ROOT),
            "metadata": str(GENERATION_META_ROOT),
            "raw": str(GENERATION_RAW_ROOT),
            "processed": str(GENERATION_PROCESSED_ROOT),
        },
        name="path",
    )
    .rename_axis("generation stage")
    .to_frame()
)
show(generation_paths)

## Bounded generated-data admission

Each batch is admitted from its terminal manifest, exact membership, hashes, sampling metadata, unit-bearing fields, Cartesian grid, finite values, and physical constraints. No output is persisted.

In [ ]:
datasets = {}
logs_all = {}

for batch_name in BATCH_NAMES:
    frame, logs = eda.dataframe.generate_eda_dataframe(
        batch_name,
        task=task,
        generated_data_root=GENERATED_DATA_ROOT,
        max_cases=MAX_CASES,
    )
    datasets[batch_name] = frame
    logs_all[batch_name] = logs

In [ ]:
summary_rows = []
for batch_name, frame in datasets.items():
    identity = frame.attrs["generated_batch_identity"]
    field_units = frame.attrs["field_units"]
    field_representations = frame.attrs["field_representations"]
    summary_rows.append(
        {
            "batch_id": batch_name,
            "admitted_cases": frame.attrs["available_case_count"],
            "materialized_cases": frame.attrs["loaded_case_count"],
            "task_id": frame.attrs["task_id"],
            "fields": ", ".join(
                f"{field} [physical unit: {field_units[field]}; stored: {field_representations[field]}]"
                for field in frame.attrs["field_names"]
            ),
            "grid_shape": frame.attrs["spatial_shape"],
            "batch_identity": identity["batch_manifest_identity_sha256"][:12],
        }
    )

summary = pd.DataFrame(summary_rows)
show(summary)

In [ ]:
for batch_name in BATCH_NAMES:
    show(Markdown(f"#### Validation log: {batch_name}"))
    show(Markdown("<br>".join(logs_all[batch_name])))

## Interactive EDA views

### Plot overview

- **1.1 Metadata statistics:** Compares derived scalar case statistics between generated batches.
- **1.2 Generation parameters:** Shows distributions of sampled generation parameters.
- **1.3 Field values:** Compares case-wise extrema and means, including EDA-only speed `|U| = sqrt(u² + v²)`.
- **2.1 Isotropic spectra:** Switches between ordered-prefix median and q10-q90 spectra and one exact case.
- **2.2 Directional spectra:** Switches between aggregate and one-case flow-y versus cross-stream-x comparisons.
- **2.3 Spectral evolution:** Compares one dataset-local case number across selected batches as independent, unpaired realizations.

Dataset, scope, case-count, and case-number controls update the selected view automatically.


In [ ]:
eda_panel = eda.panel.build_eda_panel(
    datasets=datasets,
    title="Generated-batch EDA",
)
show(eda_panel)